# 📚 Week 09 - Day 02: Next Word Prediction Using LSTM

Welcome to today's session! Today we will take our Recurrent Neural Network (RNN) knowledge to the next level by building a **Long Short-Term Memory (LSTM)** model to automatically predict the next word in a sentence. This is the exact same foundational concept behind predictive text keyboards and AI like ChatGPT!

### 🎯 Learning Objectives:
1. Understand **N-gram Sequencing** (creating step-by-step training data).
2. Learn how to build and train an **LSTM** architecture.
3. Understand how to use **Softmax** for categorical word prediction.
4. Build a streaming text generator that types out words dynamically!

In [1]:
import os
import time
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# GPU Memory Growth Setup
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('GPU Memory Growth Enabled! Safe to train.')
    except RuntimeError as e:
        print(e)

GPU Memory Growth Enabled! Safe to train.


## 1️⃣ Step 1: Download & Load the Text Corpus
We will use *The Adventures of Sherlock Holmes* as our training text. We will download it via Kaggle and automatically move it to our datasets folder.

In [4]:
import kagglehub

print('Downloading Kaggle Dataset...')
path = kagglehub.dataset_download('ronikdedhia/next-word-prediction')

# Move the dataset to our local folder
source_file = os.path.join(path, '1661-0.txt')
target_dir = r'/content/'
os.makedirs(target_dir, exist_ok=True)
target_file = os.path.join(target_dir, '1661-0.txt')

shutil.copy(source_file, target_file)
print(f'Dataset successfully moved to: {target_file}')

# Load the text
with open(target_file, 'r', encoding='utf-8') as f:
    text = f.read().lower()

print(f'Total characters in corpus: {len(text)}')
print('\n--- Preview ---')
print(text[:200])

c:\Users\DIPLAB\.conda\envs\tf-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [2]:
# Load the text
with open(r'E:\NAVTTC-AI-Course\datasets\next-word\1661-0.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

print(f'Total characters in corpus: {len(text)}')
print('\n--- Preview ---')
print(text[:200])

Total characters in corpus: 581888

--- Preview ---
﻿
project gutenberg's the adventures of sherlock holmes, by arthur conan doyle

this ebook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  you may copy it, gi


## 2️⃣ Step 2: Tokenization
Convert every unique word into an integer index.

In [4]:
corpus = text.split('\n')
corpus

['\ufeff',
 "project gutenberg's the adventures of sherlock holmes, by arthur conan doyle",
 '',
 'this ebook is for the use of anyone anywhere at no cost and with',
 'almost no restrictions whatsoever.  you may copy it, give it away or',
 're-use it under the terms of the project gutenberg license included',
 'with this ebook or online at www.gutenberg.net',
 '',
 '',
 'title: the adventures of sherlock holmes',
 '',
 'author: arthur conan doyle',
 '',
 'release date: november 29, 2002 [ebook #1661]',
 'last updated: may 20, 2019',
 '',
 'language: english',
 '',
 'character set encoding: utf-8',
 '',
 '*** start of this project gutenberg ebook the adventures of sherlock holmes ***',
 '',
 '',
 '',
 'produced by an anonymous project gutenberg volunteer and jose menendez',
 '',
 '',
 '',
 'cover',
 '',
 '',
 '',
 'the adventures of sherlock holmes',
 '',
 '',
 '',
 'by arthur conan doyle',
 '',
 '',
 '',
 'contents',
 '',
 '',
 '   i.     a scandal in bohemia',
 '   ii.    the red-head

In [3]:
corpus = text.split('\n')

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

total_words = len(tokenizer.word_index) + 1
print(f'Total unique words in vocabulary: {total_words}')

Total unique words in vocabulary: 8932


In [6]:
tokenizer.index_word

{1: 'the',
 2: 'and',
 3: 'to',
 4: 'of',
 5: 'a',
 6: 'i',
 7: '”',
 8: 'in',
 9: 'that',
 10: 'it',
 11: 'he',
 12: 'was',
 13: 'you',
 14: 'his',
 15: 'is',
 16: 'my',
 17: 'have',
 18: 'with',
 19: 'as',
 20: 'had',
 21: 'at',
 22: 'which',
 23: 'for',
 24: 'be',
 25: 'not',
 26: 'me',
 27: 'but',
 28: 'from',
 29: 'we',
 30: 'this',
 31: 'said',
 32: 'upon',
 33: 'there',
 34: 'holmes',
 35: 'him',
 36: 'so',
 37: 'her',
 38: 'she',
 39: 'all',
 40: '’',
 41: 'been',
 42: 'your',
 43: 'on',
 44: 'very',
 45: 'by',
 46: 'one',
 47: 'are',
 48: '“i',
 49: 'were',
 50: 'an',
 51: 'no',
 52: 'would',
 53: 'out',
 54: 'what',
 55: 'then',
 56: 'up',
 57: 'when',
 58: 'man',
 59: 'could',
 60: 'has',
 61: 'do',
 62: 'into',
 63: 'or',
 64: 'little',
 65: 'will',
 66: 'who',
 67: 'mr',
 68: 'if',
 69: 'some',
 70: 'down',
 71: 'see',
 72: 'now',
 73: 'our',
 74: 'should',
 75: 'may',
 76: 'am',
 77: 'us',
 78: 'over',
 79: 'they',
 80: 'can',
 81: 'more',
 82: 'think',
 83: 'about',
 84:

## 3️⃣ Step 3: Creating N-gram Sequences
This is the most critical step for Next Word Prediction!
If our sentence is `Hello how are you`, we must train the model step-by-step:
* `[Hello]` -> predict `how`
* `[Hello, how]` -> predict `are`
* `[Hello, how, are]` -> predict `you`

In [4]:
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f'Total training sequences created: {len(input_sequences)}')
print('Example Sequence:', input_sequences[2])

Total training sequences created: 101619
Example Sequence: [145, 4790, 1, 1020]


In [11]:
print('Example Sequence:', input_sequences[:5])

Example Sequence: [[145, 4790], [145, 4790, 1], [145, 4790, 1, 1020], [145, 4790, 1, 1020, 4], [145, 4790, 1, 1020, 4, 128]]


## 4️⃣ Step 4: Padding & Splitting into X and y
Pad sequences with zeros at the beginning (`pre`). Then split:
* **X**: Everything except the last word.
* **y**: The last word (One-Hot Encoded).

In [5]:
max_sequence_len = max([len(x) for x in input_sequences])
print(f'Max sequence length: {max_sequence_len}')

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

X = input_sequences[:, :-1]
labels = input_sequences[:, -1]

# y = to_categorical(labels, num_classes=total_words)
y = labels
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

Max sequence length: 20
X shape: (101619, 19)
y shape: (101619,)


## 5️⃣ Step 5: Building the LSTM Model
LSTMs are powerful because they have an internal memory that remembers context from words earlier in the sentence!

In [ ]:
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras import  regularizers
model = Sequential([
    # 1. High-capacity Embedding
        # 256-300 is the 'sweet spot' for vocab sizes under 10k.
        Embedding(total_words, 256, input_length=max_sequence_len-1),
        
        # 2. Stacked LSTM Layers
        # Layer 1 captures syntax. Layer 2 captures semantics.
        LSTM(512, return_sequences=True, dropout=0.1),

        BatchNormalization(), # Stabilizes training in deep stacks
        
        LSTM(512),
        
        # 3. Dense 'Reasoning' Layer
        Dense(1024, activation='relu', 
                     kernel_regularizer=regularizers.l2(0.001)),
        Dropout(0.4),
        
        # 4. Final Classification
        Dense(total_words, activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_4 (Embedding)     (None, 19, 256)           2286592   
                                                                 
 lstm_8 (LSTM)               (None, 19, 512)           1574912   
                                                                 
 batch_normalization_1 (Batc  (None, 19, 512)          2048      
 hNormalization)                                                 
                                                                 
 lstm_9 (LSTM)               (None, 512)               2099200   
                                                                 
 dense_5 (Dense)             (None, 1024)              525312    
                                                                 
 dropout_1 (Dropout)         (None, 1024)              0         
                                                      

## 6️⃣ Step 6: Training the Model
Let's teach the LSTM how to speak like Sherlock Holmes!
*(Note: For demonstration, we will train for a small number of epochs. For better predictions, you would train for 50-100 epochs).*

In [27]:
print('Starting training...')
history = model.fit(X, y, epochs=50, batch_size=128, verbose=1)
print('Training Complete!')

Starting training...
Epoch 1/50


794/794 [==============================] - 30s 33ms/step - loss: 6.3801 - accuracy: 0.0687
Epoch 2/50
794/794 [==============================] - 27s 34ms/step - loss: 5.7543 - accuracy: 0.1085
Epoch 3/50
794/794 [==============================] - 27s 34ms/step - loss: 5.5211 - accuracy: 0.1265
Epoch 4/50
794/794 [==============================] - 27s 34ms/step - loss: 5.3547 - accuracy: 0.1387
Epoch 5/50
794/794 [==============================] - 29s 36ms/step - loss: 5.2143 - accuracy: 0.1495
Epoch 6/50
794/794 [==============================] - 28s 35ms/step - loss: 5.0931 - accuracy: 0.1565
Epoch 7/50
794/794 [==============================] - 29s 36ms/step - loss: 4.9898 - accuracy: 0.1637
Epoch 8/50
794/794 [==============================] - 28s 35ms/step - loss: 4.8872 - accuracy: 0.1694
Epoch 9/50
794/794 [==============================] - 28s 35ms/step - loss: 4.7912 - accuracy: 0.1772
Epoch 10/50
794/794 [==============================] - 29s 36ms/step - loss: 4.6990 - accurac

## 7️⃣ Step 7: Single Next Word Prediction
Let's provide a seed text and see what word the model predicts will come next!

In [ ]:
def predict_next_word(seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted_probs, axis=-1)[0]
    predicted_word = tokenizer.index_word.get(predicted_index, '')
    return predicted_word

seed_text = 'I am going to be a happy and prosperous man'
next_word = predict_next_word(seed_text)
print(f"Seed: '{seed_text}'  -->  Predicted Next Word: '{next_word}'")

Seed: 'I am going to be a happy and prosperous'  -->  Predicted Next Word: 'man'


## 8️⃣ Step 8: ChatGPT-Style Text Generation! 🚀
Now for the fun part! We will put our prediction function inside a loop to generate an entire paragraph word-by-word. We'll use `time.sleep()` to make it stream across the screen just like an AI chatbot!

In [14]:
seed_text = 'I am going to'
next_words_to_generate = 100

print(f'\x1b[1m{seed_text}\x1b[0m', end=' ', flush=True)

for i in range(next_words_to_generate):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted_probs, axis=-1)[0]

    output_word = tokenizer.index_word.get(predicted_index, '')
    seed_text += ' ' + output_word
    if i%20==0:
        print()
    print(output_word, end=' ', flush=True)
    time.sleep(0.05)

print('\n\nGeneration Finished!')

I am going to 
be a happy and prosperous man without a hundred yards from her room and i had heard him then to 
the police doctor that we are one or two of whom we may take a few minutes to the time 
which we could not explain you to you ” he answered “for you never heard him whatever the young lady 
was a little one to my wife and to meet in the time of the opium adler ” he shrieked 
out by the door of the alpha and others i have no doubt that he will lead up i have 

Generation Finished!


To make this model "Sherlock-level" smart, you need to introduce Stochastic Sampling with a Temperature $(\tau)$ parameter. This allows the model to occasionally pick the 2nd or 3rd most likely word, adding variety and preventing loops.The probability of word $i$ is adjusted using:$$P_i = \frac{\exp(\frac{\log(y_i)}{\tau})}{\sum_j \exp(\frac{\log(y_j)}{\tau})}$$

In [17]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-7) / temperature # Add small epsilon to avoid log(0)
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

seed_text = 'Sherlock looked at the'
next_words_to_generate = 50

print(f'\x1b[1m{seed_text}\x1b[0m', end=' ', flush=True)

for i in range(next_words_to_generate):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    
    predicted_probs = model.predict(token_list, verbose=0)[0]
    
    # Use the sample function instead of argmax
    # Low temp (0.2) = Confident/Repetitive
    # High temp (1.0+) = Creative/Wild
    predicted_index = sample(predicted_probs, temperature=1)

    output_word = tokenizer.index_word.get(predicted_index, '')
    seed_text += ' ' + output_word
    
    print(output_word, end=' ', flush=True)
    if (i + 1) % 10 == 0: print() # Better readability for students

Sherlock looked at the death of words as taken on town when the other 
was just as good as my uncle and i have 
already come for you are good than it has lost 
the other thing for you ” i remarked as a 
working hypothesis that he could not invent in deep one 


In [65]:
model.save('next_word_predictor.keras')

In [6]:
from tensorflow.keras.models import load_model

model = load_model(r'E:\NAVTTC-AI-Course\Month 03\Week 09\Notebooks\next_word_predictor.keras')
